# 03 Find Project Root

## Goal

- Sometimes notebooks are executed from different folders.
- To build reliable paths, we can locate the **project root** automatically by searching upward for a marker such as `.git`.

## Idea

- If your project contains a `.git` folder, then that folder is a good marker for the project root.
- We can start at the current working directory and move upward until we find it.

In [ ]:
from pathlib import Path

def find_project_root(marker=".git"):
    path = Path.cwd()

    while path != path.parent:
        if (path / marker).exists():
            return path
        path = path.parent

    raise RuntimeError(f"Project root not found. Could not locate marker: {marker}")

In [ ]:
try:
    root = find_project_root()
    print("Project root found:")
    print(root)

    data_path = root / "data" / "countries.geojson"
    print("\nExample data path from root:")
    print(data_path)
    print("Exists:", data_path.exists())
except RuntimeError as e:
    print("Notice:")
    print(e)

## Why This Matters

- A root-finding function makes notebooks more reliable because the path logic does not depend as heavily on where the notebook happened to start.

- This becomes especially useful once students begin using:

  - `src/` layouts
  - helper libraries
  - nested notebook folders
  - bigger projects

In [ ]:
# Optional: inspect the current directory and its parents
current = Path.cwd()
print("Current directory and parents:")
for p in [current, *current.parents]:
    print(" -", p)

## Exercise A

1. What happens when `find_project_root()` is called in a project with no `.git` folder?  
   It keeps moving upward through parent directories until it reaches the filesystem root.  
   If it never finds `.git`, it raises a `RuntimeError`.

2. Name two other files or folders that could serve as a reliable project root marker.  
   Good examples are `pyproject.toml`, `setup.py`, `requirements.txt`, or `.venv`  
   (as long as the project consistently uses that marker).

3. Why is locating the root this way more robust than assuming the notebook always runs from the same folder?  
   Because notebooks can be launched from different directories, and hardcoding path assumptions breaks easily.  
   Searching upward for a marker lets the notebook adapt to where it was started.

## Exercise B

In [ ]:
from pathlib import Path

# Try a different marker that may or may not exist in this project.
for marker in ["pyproject.toml", "banana"]:
    try:
        found = find_project_root(marker=marker)
        print(f"Marker {marker!r} found at: {found}")
    except RuntimeError as e:
        print(f"Marker {marker!r} -> {e}")

**What this tells you**

- If `"pyproject.toml"` works, then this project likely uses modern Python project metadata at its root.
- If `"banana"` fails, that is expected because the marker does not exist.
- The `RuntimeError` message is useful because it tells you exactly what marker Python was trying to locate.

## Exercise C

In [ ]:
from pathlib import Path

root = find_project_root()

# Build the path from root to this module's data folder and check countries.geojson
data_file = root / "data" / "countries.geojson"

print("Looking for:", data_file)
print("Exists:", data_file.exists())

## Optional Advanced — Multiple Markers and Custom Start

In [ ]:
from pathlib import Path

def find_project_root_multi(markers=(".git", "pyproject.toml", "setup.py"), start: Path = None):
    """Return the first ancestor directory that contains any of the marker files/folders."""
    path = Path.cwd() if start is None else Path(start)

    # If start points to a file, begin from its parent directory.
    if path.is_file():
        path = path.parent

    while path != path.parent:
        for marker in markers:
            if (path / marker).exists():
                return path
        path = path.parent

    # Also check the filesystem root itself before failing.
    for marker in markers:
        if (path / marker).exists():
            return path

    raise RuntimeError(f"Project root not found. Could not locate any marker in: {markers}")

root = find_project_root_multi()
print("Root:", root)

## Check Your Understanding

1. `find_project_root()` walks upward until `path == path.parent`. What does that condition mean — when does it become true?  
   It becomes true at the filesystem root, because the root directory is its own parent.

2. You call `find_project_root()` from a notebook and get back `/project`. You then write `root / "data" / "cities.json"`. What is the full absolute path that produces?  
   `/project/data/cities.json`

3. A teammate hardcodes `Path("/Users/them/project/data/cities.json")` instead of using `find_project_root()`. What breaks when you run their notebook?  
   It breaks on your machine because that absolute path points to their personal folder layout, not yours.

---
# Summary

## Students should now understand

- what the working directory is
- how relative and absolute paths differ
- how to build paths into other folders
- how to check whether files exist
- how to locate a project root marker

## Recommended next step

Move into:

`01_JSON_GeoJSON`

because now students are ready to locate files **before** opening and inspecting them.

That tiny detail saves a lot of chaos later. A shocking amount, really.